### Criando funcao para diferenciar componentes de equacao: coeficientes e variaveis

In [25]:
# Definindo funcao que separa coeficientes e variaveis em listas
def funcao_coef_variaveis(string_equacao):
    # isto esta ok, e consiste em tirar os sinais positivos ou negativos dos coeficientes
    lista_sinal_coeficientes = []
    for elemento in string_equacao:
        if elemento == '+':
            lista_sinal_coeficientes.append(elemento)
        if elemento == '-':
            lista_sinal_coeficientes.append(elemento)
    
    # isto esta ok, e consiste em tirar os coeficientes e que a mesma dimensao da lista dos sinais concida com os coeficientes
    novo_string = string_equacao
    for i in range(1,100):
        for sep in ['x_', 's_', 'e_']:
            novo_string = novo_string.replace(f'{sep}{i}', 'o')
    novo_string = novo_string.replace(' ', '').replace('+', '').replace('-', '')    
    lista_coeficiente = novo_string.split('o')    
    if '' in lista_coeficiente:
        lista_coeficiente.remove('')
    elif ' ' in lista_coeficiente:
        lista_coeficiente.remove(' ')        
    
    # aqui separa pelos segmentos que contem coeficientes e x_ ou s_ ou e_
    separando_equacao = string_equacao.replace('-', '+').split('+')
    if '' in separando_equacao:
        separando_equacao.remove('')
    elif ' ' in separando_equacao:
        separando_equacao.remove(' ')
    dicionario_c={}

    # se utiliza a separacao em partes para criar um dicionario que outorgue coeficientes as respectivas variaveis 
    # print('teste', separando_equacao)
    for i, elemento in enumerate(separando_equacao):
        if 'x' in elemento:
            coeficiente, variavel = elemento.split('x')
            dicionario_c[f'x{variavel.strip()}'] = float(f'{lista_sinal_coeficientes[i]}{coeficiente.strip()}')
        elif 's' in elemento:
            coeficiente, variavel = elemento.split('s')
            dicionario_c[f's{variavel.strip()}'] = float(f'{lista_sinal_coeficientes[i]}{coeficiente.strip()}')
        elif 'e' in elemento:
            coeficiente, variavel = elemento.split('e')
            dicionario_c[f'e{variavel.strip()}'] = float(f'{lista_sinal_coeficientes[i]}{coeficiente.strip()}')
        elif 'a' in elemento:
            coeficiente, variavel = elemento.split('a')
            dicionario_c[f'a{variavel.strip()}'] = float(f'{lista_sinal_coeficientes[i]}{coeficiente.strip()}')
        
    return dicionario_c

### Funcao para criar matriz de restricoes A, considerando variaveis folga, excedentes, e artificiais

In [26]:
def matriz_restricoes(lista_restricoes):
        # caso <=, entao adicionar s_i (variavel de folga)
    i = 1
    tupla_matriz_A = ()
    tupla_coef_b = ()
    for restricao in lista_restricoes:
        if '<=' in restricao:
            equacao_restricao, coef_b = restricao.split('<=') 
            equacao_restricao = equacao_restricao + f'+ 1s_{i}'
        
        # caso >=, entao adicionar e_i (variavel excedente) e tambem a_i (variavel artificial)
        elif '>=' in restricao:
            equacao_restricao, coef_b = restricao.split('>=')
            equacao_restricao = equacao_restricao + f'- 1e_{i} '
            i += 1
            equacao_restricao = equacao_restricao + f'+ 1a_{i} '
        
        # caso ==, entao adicionar a_i
        elif '==' in restricao:
            equacao_restricao, coef_b = restricao.split('==')
            equacao_restricao = equacao_restricao + f'+ 1a_{i} '

        dicionario_restricao = funcao_coef_variaveis(equacao_restricao)  
        tupla_matriz_A += (dicionario_restricao, )
        tupla_coef_b += (float(coef_b.strip()), )
        
        i += 1
    
    return tupla_matriz_A, tupla_coef_b

### Funcao conversao dicionario de restricao em lista de restricoes

In [27]:
def funcao_conversao_dicionario_restricao_em_lista_restricao(tupla_restricoes, lista_todas_variaveis):
    lista_matriz_A = []
    lista_restricao = []
    for dicionario_restricao in tupla_restricoes:
        lista_restricao = []
        for variavel in lista_todas_variaveis:
            if variavel in dicionario_restricao:
                lista_restricao.append(dicionario_restricao[variavel])
            else:   
                lista_restricao.append(0)
        lista_matriz_A.append(lista_restricao)
    return lista_matriz_A

### Funcao conversao dicionario de funcao em lista de funcao

In [28]:
def funcao_conversao_dicionario_funcaoobjetivo_em_lista_funcaoobjetivo(dicionario_fo, lista_todas_variaveis):
    lista_fo = []
    for variavel in lista_todas_variaveis:
        if variavel in dicionario_fo:
            lista_fo.append(dicionario_fo[variavel])
        else:   
            lista_fo.append(0)
    return lista_fo

### Funcao para determinar se ha variaveis artificias (o que levaria à Fase 1)

In [29]:
def funcao_caso_haja_variaveis_artificias(lista_todas_variaveis):
    variaveis_artificiais = []
    teste_a = 'a_'
    for i in range(1, 20):
        variacao_teste_a = f'{teste_a}{i}'
        if variacao_teste_a in lista_todas_variaveis:
            variaveis_artificiais.append(variacao_teste_a)
    if len(variaveis_artificiais) > 0:
        print('Existem variaveis artificiais:\n', variaveis_artificiais) 
        return True
    else:
        return False
    

### Funcao para calculo de Z_j e Z_j-C_j

In [30]:
def funcao_calculo_Z_j_e_Z_j_menos_C_j(lista_todas_variaveis, lista_matriz_A, coeficiente_base_tableau_C_j, vetor_coeficientes_C_B):
    
    vetor_Z_j = []
    vetor_C_j_menos_Z_j = []
    for i, variavel in enumerate(lista_todas_variaveis):
        Z = 0
        for k, elemento in enumerate(vetor_coeficientes_C_B):
            valor_A_j = lista_matriz_A[k][i] 
            Z += elemento*valor_A_j
        vetor_Z_j.append(Z)
        
        C_j_menos_Z_j = 0    
        C_j_menos_Z_j = coeficiente_base_tableau_C_j[i]-vetor_Z_j[i]
        vetor_C_j_menos_Z_j.append(C_j_menos_Z_j)
    print('vetor_Z_j',  vetor_Z_j)    
    return vetor_Z_j, vetor_C_j_menos_Z_j

### Funcao calculo iteracoes

In [31]:
def funcao_calculo_iteracoes(lista_todas_variaveis, coeficiente_base_tableau_C_j, lista_matriz_A, vetor_variaveis_C_B, vetor_coeficientes_C_B, lista_coef_b, coluna_pivo):
    indice_coluna_pivo = lista_todas_variaveis.index(coluna_pivo) # lembrando que eh o valor da coluna que substitui o valor da fila
    indice_fila_pivo = vetor_variaveis_C_B.index(coluna_pivo)
    valor_pivo = lista_matriz_A[indice_fila_pivo][indice_coluna_pivo]
    
    for j, elemento_fila in enumerate(lista_matriz_A[indice_fila_pivo]):
        valor_pivo_substituir = float(elemento_fila / valor_pivo)
        lista_matriz_A[indice_fila_pivo][j] = valor_pivo_substituir
    lista_coef_b[indice_fila_pivo] = lista_coef_b[indice_fila_pivo] / valor_pivo

    for i, fila_matriz in enumerate(lista_matriz_A):        
        # Tratando matriz A
        if i != indice_fila_pivo:
            valor_a_eliminar = fila_matriz[indice_coluna_pivo]
            for j, elemento_fila in enumerate(fila_matriz):
                valor_a_substituir_em_fila = (lista_matriz_A[indice_fila_pivo][j] * (-valor_a_eliminar) ) + lista_matriz_A[i][j]
                lista_matriz_A[i][j] = valor_a_substituir_em_fila
        # Tratando vetor b
            valor_a_substituir_em_vetor_b = lista_coef_b[indice_fila_pivo] * (-valor_a_eliminar) + lista_coef_b[i] 
            lista_coef_b[i] = valor_a_substituir_em_vetor_b
    
    return lista_matriz_A, lista_coef_b, vetor_coeficientes_C_B

### Funcao valores C_j e vetor C_B

In [32]:
# 1 Define-se os valores de todas as variaveis base do tableau C_j (consideram-se todas as variaveis) 
def funcao_C_j_e_C_B(lista_todas_variaveis, lista_fo, M):
    coeficiente_base_tableau_C_j = []
    for i, variavel in enumerate(lista_todas_variaveis):
        if 'x' in variavel:
            coeficiente_base_tableau_C_j.append(lista_fo[i])
        elif 'a' in variavel:
            coeficiente_base_tableau_C_j.append(-M)
        else:
            coeficiente_base_tableau_C_j.append(0)
    print('variaveis_base_tableau_C_j: ', lista_todas_variaveis)
    print('coeficiente_base_tableau_C_j: ', coeficiente_base_tableau_C_j)

    # 2. Cria-se vetor C_B que contenha as variaveis s_i, e a_i
    vetor_variaveis_C_B = []  
    for variavel in lista_todas_variaveis:
        if 's' in variavel:
            vetor_variaveis_C_B.append(variavel)
        elif 'a' in variavel:
            vetor_variaveis_C_B.append(variavel)
    print('vetor_variaveis_C_B: ', vetor_variaveis_C_B)

    # 3 Define-se o vetor valores C_B baseado nos valores de s_i, e a_i
    vetor_coeficientes_C_B = []
    for i, variavel in enumerate(lista_todas_variaveis):
        if variavel in vetor_variaveis_C_B:
            vetor_coeficientes_C_B.append(coeficiente_base_tableau_C_j[i])
    print('vetor_coeficientes_C_B: ', vetor_coeficientes_C_B)
    return coeficiente_base_tableau_C_j, vetor_variaveis_C_B, vetor_coeficientes_C_B

### Funcao loop (while) até C_j menos Z_j ser <=0

In [33]:
def funcao_loop_C_jmenosZ_j_ate_menor_a_0(lista_matriz_A, lista_todas_variaveis, vetor_variaveis_C_B, vetor_coeficientes_C_B, coeficiente_base_tableau_C_j, vetor_C_j_menos_Z_j, lista_coef_b):
    lista_C_j_Z_j_evitar_infinito = []
    k=0
    while any(e > 0 for e in vetor_C_j_menos_Z_j):
        print('-------------------------------------------------------------------------------------')
        print(f'# Inicia loop n°{k+1}')
        # Caso se calcula a base que entra-sai, entao comeca-se por definir o maior valor de C_j-Z_j, e define-se a coluna a utilizar (.index()) e a variavel
        # print('vetor_C_j_menos_Z_j' ,vetor_C_j_menos_Z_j)
        
        # ALTERACAO PARA EVITAR LOOP INFINITO - TESTE
        print('vetor_C_j_menos_Z_j', vetor_C_j_menos_Z_j)
        lista_C_j_Z_j_evitar_infinito.append(tuple(vetor_C_j_menos_Z_j))
        
        if len(set(lista_C_j_Z_j_evitar_infinito)) == len(lista_C_j_Z_j_evitar_infinito):
            maior_valor_C_j_Z_j = max(vetor_C_j_menos_Z_j)
        else:
            print('Parece que entramos em loop infinito, vamos corrijir para C_j-Z_j!')
            vetor_C_j_menos_Z_j[vetor_C_j_menos_Z_j.index(max(vetor_C_j_menos_Z_j))] = vetor_C_j_menos_Z_j[vetor_C_j_menos_Z_j.index(max(vetor_C_j_menos_Z_j))] - 1
            maior_valor_C_j_Z_j = max(vetor_C_j_menos_Z_j)
        print('vetor_C_j_menos_Z_j', vetor_C_j_menos_Z_j)

        indice_maior_valor_C_j_Z_j = vetor_C_j_menos_Z_j.index(maior_valor_C_j_Z_j) # INDICE COLUNA PIVO
        # print('indice_maior_valor_C_j_Z_j', indice_maior_valor_C_j_Z_j)
        coluna_pivo = lista_todas_variaveis[indice_maior_valor_C_j_Z_j]

        # Depois, se calcula a razao entre a coluna pivo e o seu correspondente b (vetor_b ou lista_coef_b)
            # Caso o valor da razao sea positivo, eh considerado no vetor razao
        vetor_razao = []
        for i, dividendo in enumerate(lista_coef_b):
            divisor = lista_matriz_A[i][indice_maior_valor_C_j_Z_j]
            if divisor == 0:
                vetor_razao.append(0)
            else:
                razao = dividendo / divisor
                if razao >= 0:
                    vetor_razao.append(razao)
                else:
                    vetor_razao.append(0)
            # O valor minimo do vetor razao vai definir qual elemento sai da variaveis base, e qual elemento entra 
        vetor_razao_sem0 = vetor_razao[:]
        if 0 in vetor_razao_sem0:
            vetor_razao_sem0.remove(0)
        print('vetor_razao', vetor_razao)
        menor_valor_razao = min(vetor_razao_sem0)    
        print(menor_valor_razao)
        indice_menor_valor_razao = vetor_razao.index(menor_valor_razao)
        fila_pivo =  vetor_variaveis_C_B[indice_menor_valor_razao] 
            # Se troca a variavel que sai pela que entra
        print('vetor variaveis C_B:', vetor_variaveis_C_B)
        print('vetor_coeficientes_C_B:', vetor_coeficientes_C_B)    
        vetor_variaveis_C_B[vetor_variaveis_C_B.index(fila_pivo)] = coluna_pivo
            # Atualizando vetor C_B
        for i, variavel_C_B in enumerate(vetor_variaveis_C_B):
            # if variavel_C_B in lista_todas_variaveis:
            indice_a_ser_considerado = lista_todas_variaveis.index(variavel_C_B)
            valor_a_ser_substituido_em_C_B = coeficiente_base_tableau_C_j[indice_a_ser_considerado]       
            vetor_coeficientes_C_B[i] = valor_a_ser_substituido_em_C_B

        print('novo vetor variaveis C_B:', vetor_variaveis_C_B)
        print('novo vetor C_B:', vetor_coeficientes_C_B)
        print('matriz_A:', lista_matriz_A)
        print('vetor b:', lista_coef_b)
        print('\n     # Inicia iteracao')
        lista_matriz_A, lista_coef_b, vetor_coeficientes_C_B = funcao_calculo_iteracoes(
            lista_todas_variaveis, 
            coeficiente_base_tableau_C_j, 
            lista_matriz_A, 
            vetor_variaveis_C_B, 
            vetor_coeficientes_C_B, 
            lista_coef_b,
            coluna_pivo
            )

        print('matriz A tratada:', lista_matriz_A)
        print('vetor_b tratado', lista_coef_b)    
        print('\n     # Inicia calculo Z_j_e_Z_j_menos_C_j')
        vetor_Z_j, vetor_C_j_menos_Z_j = funcao_calculo_Z_j_e_Z_j_menos_C_j(lista_todas_variaveis, lista_matriz_A, coeficiente_base_tableau_C_j, vetor_coeficientes_C_B)

        # print('vetor_Z_j', vetor_Z_j) 
        print('vetor_C_j_menos_Z_j', vetor_C_j_menos_Z_j)
        if k==10:
            break
        k+=1
    otimo_fo = sum(elemento_C_B*lista_coef_b[i] for i, elemento_C_B in enumerate(vetor_coeficientes_C_B) )

    texto_fim_loop = f'''-------------------------------------------------------
    -------------------------------------------------------
    Fim do loop:
    otimo da fo: {-otimo_fo}
    valores f.o. fase1: {coeficiente_base_tableau_C_j}
    matriz_A: {lista_matriz_A}
    vetor_b: {lista_coef_b}'''   
    [print(lista) for lista in lista_C_j_Z_j_evitar_infinito]
    return  print(texto_fim_loop)


## **Tratamento para obter os numeros dos vetores e matrizes**

In [34]:
# Definir a f.o.: min. ou max. e digite a funcao
fo = 'max. + 0x_1 +  0x_3 + 0x_2 + 0x_4 + 1x_5'
fo_min_max, fo_equacao = fo.split('.')
fo_equacao.replace(' ', '')

# Obter variaveis e coeficientes separados numa funcao
dicionario_fo =  funcao_coef_variaveis(fo_equacao)

# Definir as restricoes, tornar os coeficientes a direita positivos
# g_0 = '+ 3000x_1 + 20000x_2 + 30000x_3 + 10000x_4 + 37777x_5 <= 137777'
# h_1 = '+ 1x_1 + 1x_2 + 1x_3 + 1x_4 + 5x_5 <= 25'
# g_2 = '+ 20x_1 + 5x_2 + 10x_3 + 2x_4 + 250x_5 <= 500'
# g_3= '+ 10x_1 + 20x_2 + 20x_3 + 15x_4 - 50x_5 >= 50'
# g_4= '+ 0x_1 + 0x_2 + 0x_3 + 0x_4 + 1x_5 <= 1'
# g_5= '+ 0x_1 + 0x_2 + 0x_3 + 0x_4 + 1x_5 >= 0'
# lista_restricoes = [
#     g_0,
#     h_1,
#     g_2, 
#     g_3,
#     g_4,
#     # g_5
#     ]

g_0 = '+ 3000x_1  + 30000x_3 + 20000x_2 + 10000x_4 + 37777x_5 <= 137777'
h_1 = '+ 1x_1 +  1x_3 + 1x_2 + 1x_4 + 5x_5 <= 25'
g_2 = '+ 20x_1 +  10x_3 + 5x_2 + 2x_4 + 250x_5 <= 500'
g_3= '+ 10x_1  + 20x_3 + 20x_2 +  15x_4 - 50x_5 >= 50'
g_4= '+ 0x_1  + 0x_3 + 0x_2 + 0x_4 + 1x_5 <= 1'
g_5= '+ 0x_1  + 0x_3  + 0x_2 + 0x_4 + 1x_5 >= 0'
lista_restricoes = [
    g_0,
    h_1,
    g_2, 
    g_3,
    g_4,
    # g_5
    ]

# Procesamento para obtencao da matriz em forma de tupla(dicionarios)
tupla_restricoes, tupla_coef_b = matriz_restricoes(lista_restricoes)

# Organizando tudo em vetor
    # Tirar todas as chaves da tupla e organizar em order
lista_todas_variaveis = []
for dicionario in tupla_restricoes:
    lista_chaves = list(dicionario.keys())
    for chave in lista_chaves:
        if chave not in lista_todas_variaveis:
            lista_todas_variaveis.append(chave)
    
# Organizar cada restricao em forma de vetor, adicionar os 0s caso nao haja coeficiente
lista_matriz_A = funcao_conversao_dicionario_restricao_em_lista_restricao(tupla_restricoes, lista_todas_variaveis)

# Printar funcao objetivo, restricoes, e vetor b    
lista_coef_b = list(tupla_coef_b)
lista_fo = funcao_conversao_dicionario_funcaoobjetivo_em_lista_funcaoobjetivo(dicionario_fo, lista_todas_variaveis)

print('variaveis:', lista_todas_variaveis)
print('vetor fo:', lista_fo)
print('matriz A:' , lista_matriz_A)
print('vetor b:', lista_coef_b)

variaveis: ['x_1', 'x_3', 'x_2', 'x_4', 'x_5', 's_1', 's_2', 's_3', 'e_4', 'a_5', 's_6']
vetor fo: [0.0, 0.0, 0.0, 0.0, 1.0, 0, 0, 0, 0, 0, 0]
matriz A: [[3000.0, 30000.0, 20000.0, 10000.0, 37777.0, 1.0, 0, 0, 0, 0, 0], [1.0, 1.0, 1.0, 1.0, 5.0, 0, 1.0, 0, 0, 0, 0], [20.0, 10.0, 5.0, 2.0, 250.0, 0, 0, 1.0, 0, 0, 0], [10.0, 20.0, 20.0, 15.0, -50.0, 0, 0, 0, -1.0, 1.0, 0], [0.0, 0.0, 0.0, 0.0, 1.0, 0, 0, 0, 0, 0, 1.0]]
vetor b: [137777.0, 25.0, 500.0, 50.0, 1.0]


## **Definindo o tableau para minimizar a funcao objetivo**
1. Definir base do tableau: Escolher as bases baseado nas folgas com indice nao negativo (variaveis >=0)
    - Utilizar a lista de variaveis (x, s, e, a) e fazer uma leitura de cada fila de restricao.
    - Caso a variavel seja a, e, ou s, e o coeficiente respectivo for positivo, entao adicionar essa variavel como base.
2. Calcular $C_j - Z_j$ utilizando:
    - na Fase 1, a maximizacao da funcao $-W = \sum_{i=1}^{n} -A_i$
    - na Fase 2, a maximizacao da funcao objeito original

In [35]:
fase1_true_fase2_false = funcao_caso_haja_variaveis_artificias(lista_todas_variaveis)
if fase1_true_fase2_false == True:
    M=10

coeficiente_base_tableau_C_j, vetor_variaveis_C_B, vetor_coeficientes_C_B = funcao_C_j_e_C_B(lista_todas_variaveis, lista_fo, M)

## AQUI DEVE COMECAR A FUNCAO PARA Z_j e C_j-Z_j
vetor_Z_j, vetor_C_j_menos_Z_j = funcao_calculo_Z_j_e_Z_j_menos_C_j(lista_todas_variaveis, lista_matriz_A, coeficiente_base_tableau_C_j, vetor_coeficientes_C_B)
print('vetor_C_j_menos_Z_j', vetor_C_j_menos_Z_j)
## AQUI DEVE COMECAR O LOOP ATE C_j-Z_j <= 0
funcao_loop_C_jmenosZ_j_ate_menor_a_0(lista_matriz_A, lista_todas_variaveis, vetor_variaveis_C_B, vetor_coeficientes_C_B, coeficiente_base_tableau_C_j, vetor_C_j_menos_Z_j, lista_coef_b)


Existem variaveis artificiais:
 ['a_5']
variaveis_base_tableau_C_j:  ['x_1', 'x_3', 'x_2', 'x_4', 'x_5', 's_1', 's_2', 's_3', 'e_4', 'a_5', 's_6']
coeficiente_base_tableau_C_j:  [0.0, 0.0, 0.0, 0.0, 1.0, 0, 0, 0, 0, -10, 0]
vetor_variaveis_C_B:  ['s_1', 's_2', 's_3', 'a_5', 's_6']
vetor_coeficientes_C_B:  [0, 0, 0, -10, 0]
vetor_Z_j [-100.0, -200.0, -200.0, -150.0, 500.0, 0.0, 0.0, 0.0, 10.0, -10.0, 0.0]
vetor_C_j_menos_Z_j [100.0, 200.0, 200.0, 150.0, -499.0, 0.0, 0.0, 0.0, -10.0, 0.0, 0.0]
-------------------------------------------------------------------------------------
# Inicia loop n°1
vetor_C_j_menos_Z_j [100.0, 200.0, 200.0, 150.0, -499.0, 0.0, 0.0, 0.0, -10.0, 0.0, 0.0]
vetor_C_j_menos_Z_j [100.0, 200.0, 200.0, 150.0, -499.0, 0.0, 0.0, 0.0, -10.0, 0.0, 0.0]
vetor_razao [4.5925666666666665, 25.0, 50.0, 2.5, 0]
2.5
vetor variaveis C_B: ['s_1', 's_2', 's_3', 'a_5', 's_6']
vetor_coeficientes_C_B: [0, 0, 0, -10, 0]
novo vetor variaveis C_B: ['s_1', 's_2', 's_3', 'x_3', 's_6']
nov

In [36]:
texto_matriz_A_editar = '[[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.9999999999999999], [0.26, 0.0, -0.19999999999999996, 0.0, 0.0, 1.9999999999999998e-05, 1.0, 0.0, 0.08000000000000002, -0.08000000000000002, -9.75554], [20.28, 0.0, -0.6000000000000014, 0.0, 0.0, -0.00043999999999999985, 0.0, 1.0, -0.16000000000000014, 0.16000000000000014, -225.37812], [-0.22000000000000003, 1.0, 0.4, 0.0, 0.0, 5.9999999999999995e-05, 0.0, 0.0, 0.04, -0.04, -4.26662], [0.96, 0.0, 0.8, 1.0, 0.0, -7.999999999999999e-05, 0.0, 0.0, -0.12, 0.12, 9.02216]]'
# texto_matriz_A_editar = list(texto_matriz_A_editar)

texto_matriz_A_editar=texto_matriz_A_editar.replace('],', '\n').replace('[','').replace(']','').replace(',', '')
print(texto_matriz_A_editar)

0.0 0.0 0.0 0.0 1.0 0.0 0.0 0.0 0.0 0.0 0.9999999999999999
 0.26 0.0 -0.19999999999999996 0.0 0.0 1.9999999999999998e-05 1.0 0.0 0.08000000000000002 -0.08000000000000002 -9.75554
 20.28 0.0 -0.6000000000000014 0.0 0.0 -0.00043999999999999985 0.0 1.0 -0.16000000000000014 0.16000000000000014 -225.37812
 -0.22000000000000003 1.0 0.4 0.0 0.0 5.9999999999999995e-05 0.0 0.0 0.04 -0.04 -4.26662
 0.96 0.0 0.8 1.0 0.0 -7.999999999999999e-05 0.0 0.0 -0.12 0.12 9.02216


In [37]:
texto_matriz_A_editar2 = '[[-7000.0, 0.0, 10000.0, -5000.0, 87777.0, 1.0, 0.0, 0.0, 1000.0, -1000.0, 0.0], [0.5, 0.0, 0.0, 0.25, 7.5, 0.0, 1.0, 0.0, 0.05, -0.05, 0.0], [17.5, 0.0, 5.0, -1.75, 262.5, 0.0, 0.0, 1.0, 0.25, -0.25, 0.0], [0.5, 1.0, 1.0, 0.75, -2.5, 0.0, 0.0, 0.0, -0.05, 0.05, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]]'

texto_matriz_A_editar2=texto_matriz_A_editar2.replace('],', '\n').replace('[','').replace(']','').replace(',', '')
print(texto_matriz_A_editar2)


-7000.0 0.0 10000.0 -5000.0 87777.0 1.0 0.0 0.0 1000.0 -1000.0 0.0
 0.5 0.0 0.0 0.25 7.5 0.0 1.0 0.0 0.05 -0.05 0.0
 17.5 0.0 5.0 -1.75 262.5 0.0 0.0 1.0 0.25 -0.25 0.0
 0.5 1.0 1.0 0.75 -2.5 0.0 0.0 0.0 -0.05 0.05 0.0
 0.0 0.0 0.0 0.0 1.0 0.0 0.0 0.0 0.0 0.0 1.0
